# Parameter Files

The previous notebooks build models directly in Python, which is ideal when you want to inspect each object. TauREx can also be configured declaratively through parameter files, and that mode is still useful for reproducible runs, batch jobs, and sharing a compact setup with collaborators.

This notebook reads like a translation exercise: we write a small text configuration, then ask TauREx to turn it back into the same kinds of model objects you have already seen in Python.

In [1]:
from pathlib import Path
from _shared import CIA_DIR, XSEC_DIR, ensure_opacity_data

ensure_opacity_data(download=False)
output_path = Path('input.par')

print(f'Cross-sections will be read from: {XSEC_DIR}')
print(f'CIA files will be read from: {CIA_DIR}')
print(f'Parameter file will be written to: {output_path.resolve()}')

Cross-sections will be read from: /home/simone/Dropbox/eScience_projects/TauRex/taurex3/examples/tmp/xsec
CIA files will be read from: /home/simone/Dropbox/eScience_projects/TauRex/taurex3/examples/tmp/cia
Parameter file will be written to: /home/simone/Dropbox/eScience_projects/TauRex/taurex3/examples/notebooks/input.par


The first cell is mostly bookkeeping, but it teaches an important point: parameter files still depend on external opacity paths, so reproducibility means recording both the model configuration and the data locations it expects.

In [2]:
input_par = f'''[Global]
xsec_path = {XSEC_DIR}
cia_path = {CIA_DIR}

[Chemistry]
chemistry_type = taurex
fill_gases = H2,He
ratio = 0.1741

    [[H2O]]
    gas_type = constant
    mix_ratio = 1e-3

[Temperature]
profile_type = isothermal
T = 2000

[Pressure]
profile_type = Simple
atm_min_pressure = 1e-5
atm_max_pressure = 1e5
nlayers = 100

[Planet]
planet_type = Simple
planet_mass = 0.74
planet_radius = 1.38

[Star]
star_type = blackbody
temperature = 6117
radius = 1.16

[Model]
model_type = transmission

    [[Absorption]]
'''

output_path.write_text(input_par)
print(output_path.read_text())

[Global]
xsec_path = /home/simone/Dropbox/eScience_projects/TauRex/taurex3/examples/tmp/xsec
cia_path = /home/simone/Dropbox/eScience_projects/TauRex/taurex3/examples/tmp/cia

[Chemistry]
chemistry_type = taurex
fill_gases = H2,He
ratio = 0.1741

    [[H2O]]
    gas_type = constant
    mix_ratio = 1e-3

[Temperature]
profile_type = isothermal
T = 2000

[Pressure]
profile_type = Simple
atm_min_pressure = 1e-5
atm_max_pressure = 1e5
nlayers = 100

[Planet]
planet_type = Simple
planet_mass = 0.74
planet_radius = 1.38

[Star]
star_type = blackbody
temperature = 6117
radius = 1.16

[Model]
model_type = transmission

    [[Absorption]]



When you read the file below, try matching each block to the corresponding Python object from the earlier notebooks. This makes parameter files much less mysterious: they are not a second modelling system, just a textual description of the same components.

A useful way to read this file is to match sections to the Python objects from earlier notebooks: `Temperature` becomes a temperature profile, `Pressure` becomes a pressure grid, `Chemistry` defines the gas mixture, and `Model` chooses the forward-model geometry plus contributions.

In [3]:
from taurex.parameter import ParameterParser

parser = ParameterParser()
parser.read(str(output_path))
parser.setup_globals()
model = parser.generate_appropriate_model()

print(type(model).__name__)
print('Contributions:', [c.name for c in model.contribution_list])

TransmissionModel
Contributions: ['Absorption']


In [4]:
print('Temperature object:', type(model.temperature).__name__)
print('Chemistry object:', type(model.chemistry).__name__)
print('Pressure object:', type(model.pressure).__name__)

Temperature object: Isothermal
Chemistry object: TaurexChemistry
Pressure object: SimplePressureProfile
